# Домашнее задание №9: Векторные БД и Бенчмаркинг FAISS

1. Взять набор документов (корпус PubMed) и построить на его основе векторную базу данных FAISS.
2. Проиндексировать с помощью модели SentenceTransformer (из MTEB).
3. Замерить, как скорость работы меняется в зависимости от настроек индекса (FlatL2, IVF, HNSW), и сделать ключевые выводы.

In [8]:
import os
import gc
import time
import logging
import warnings
import numpy as np
import pandas as pd
import faiss
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")
np.random.seed(42)

# Подавление предупреждений токенизаторов
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
import transformers
transformers.logging.set_verbosity_error()
logging.getLogger("transformers.tokenization_utils_base").setLevel(logging.ERROR)

def flush_memory():
    """Принудительная очистка VRAM и RAM"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, 'ipc_collect'):
            torch.cuda.ipc_collect()

## 1. Сбор корпуса из PubMed и подготовка Golden Set

Для тестирования нашей векторной БД в режиме "поиска иголки в стоге сена" мы скачиваем:
1. **Фон ("сено"):** 10 000 статей по инфекционным заболеваниям.
2. **Целевые данные ("иголки"):** 400 статей по опорно-двигательному аппарату (MSK).

Чтобы оценить качество векторного поиска, мы генерируем вопросы по целевым статьям (Golden Set). Весь датасет из 400 статей `pubmed_msk.csv` был загружен в **Gemini 3.1 Pro preview** для автоматической генерации сложных вопросов, требующих нахождения конкретной статьи. Результат сохранен в формате JSON.

In [9]:
from Bio import Entrez, Medline

Entrez.email = 'your_email@example.com'

# 1. Запрос для инфекционных заболеваний (фон / "сено")
PUBMED_QUERY_INFECTIONS = """
(
  "Infectious Disease Medicine"[MeSH Terms]
  OR "Communicable Diseases"[MeSH Terms]
  OR "infectious disease*"[Title/Abstract]
  OR infection*[Title/Abstract]
)
AND (
  randomized[Title/Abstract]
  OR trial[Title/Abstract]
  OR "systematic review"[Title/Abstract]
  OR "meta-analysis"[Publication Type]
)
AND english[Language]
""".strip()

# 2. Запрос для опорно-двигательного аппарата (целевые статьи / "иголки")
PUBMED_QUERY_MSK = """
(
  "Musculoskeletal Diseases"[MeSH Terms]
  OR musculoskeletal[Title/Abstract]
  OR orthopedic*[Title/Abstract]
  OR "bone disease*"[Title/Abstract]
  OR "joint disease*"[Title/Abstract]
)
AND (
  randomized[Title/Abstract]
  OR trial[Title/Abstract]
  OR "systematic review"[Title/Abstract]
  OR "meta-analysis"[Publication Type]
)
AND english[Language]
""".strip()

def download_pubmed_data(query, retmax, filename):
    if os.path.exists(filename):
        print(f"Файл {filename} уже существует, пропускаем скачивание.")
        return pd.read_csv(filename)
        
    print(f"Поиск по запросу, ожидаемое количество: {retmax}")
    try:
        with Entrez.esearch(db="pubmed", term=query, retmax=retmax, sort="relevance") as h:
            rec = Entrez.read(h)
        pmids = rec.get("IdList", [])
        print(f"Найдено PMID: {len(pmids)}")
        
        if not pmids:
            return pd.DataFrame()
            
        print("Скачивание данных... Это может занять некоторое время.")
        batch_size = 500
        records = []
        for start in tqdm(range(0, len(pmids), batch_size), desc=f"Скачивание батчей для {filename}"):
            end = min(len(pmids), start + batch_size)
            batch_pmids = pmids[start:end]
            try:
                with Entrez.efetch(db="pubmed", id=",".join(batch_pmids), rettype="medline", retmode="text") as h:
                    records.extend(list(Medline.parse(h)))
            except Exception as e:
                print(f"Ошибка при скачивании батча: {e}")
            
        rows = []
        for rec in records:
            pmid = str(rec.get('PMID', '')).strip()
            abstract = rec.get('AB', '')
            if not pmid or not abstract: continue
            rows.append({
                'pmid': pmid,
                'title': str(rec.get('TI', '')).strip(),
                'abstract': str(abstract).strip(),
                'mesh_terms': '; '.join(rec.get('MH', [])[:15]),
                'url': f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/'
            })
            
        df_corpus = pd.DataFrame(rows).drop_duplicates(subset=['pmid'])
        os.makedirs("data", exist_ok=True)
        df_corpus.to_csv(filename, index=False)
        print(f"Сохранено {len(df_corpus)} уникальных документов в {filename}\n")
        return df_corpus
    except Exception as e:
        print(f"Ошибка при работе с PubMed API: {e}")
        return pd.DataFrame()

# Скачиваем 10 000 статей по инфекциям
df_infections = download_pubmed_data(PUBMED_QUERY_INFECTIONS, 10000, "data/pubmed_infections.csv")

# Скачиваем 400 статей по опорно-двигательному аппарату
df_msk = download_pubmed_data(PUBMED_QUERY_MSK, 400, "data/pubmed_msk.csv")

# Объединяем датасеты
if not df_infections.empty and not df_msk.empty:
    df_combined = pd.concat([df_infections, df_msk]).drop_duplicates(subset=['pmid']).reset_index(drop=True)
    df_combined.to_csv("data/pubmed_combined_corpus.csv", index=False)
    print(f"Итоговый объединенный датасет сохранен. Всего документов: {len(df_combined)}")

Файл data/pubmed_infections.csv уже существует, пропускаем скачивание.
Файл data/pubmed_msk.csv уже существует, пропускаем скачивание.
Итоговый объединенный датасет сохранен. Всего документов: 10229


In [10]:
import json

## 1.1 Загрузка корпуса и подготовка данных
DATA_PATH = "data/pubmed_combined_corpus.csv"
GOLDEN_JSON_PATH = "data/pubmed_golden_msl.json"

# Загружаем корпус (сено + иголки)
try:
    df = pd.read_csv(DATA_PATH)
    df["title"] = df["title"].fillna("")
    df["abstract"] = df["abstract"].fillna("")
    df["text_full"] = df["title"] + " " + df["abstract"]
except FileNotFoundError:
    print(f"Файл {DATA_PATH} не найден. Сначала выполните скачивание.")
    df = pd.DataFrame()

# Конвертация и загрузка Golden Set из JSON, сгенерированного Gemini 3.1 Pro
try:
    with open(GOLDEN_JSON_PATH, 'r', encoding='utf-8') as f:
        golden_data = json.load(f)
    golden_set = golden_data.get('items', [])
    
    # Опционально сохраняем в CSV для совместимости, если нужно
    pd.DataFrame(golden_set).to_csv("data/golden_musculoskeletal.csv", index=False)
    
    print(f"Загружен Golden Set (из JSON): {len(golden_set)} вопросов.")
except FileNotFoundError:
    print(f"ВНИМАНИЕ: Файл {GOLDEN_JSON_PATH} не найден.")
    golden_set = []

# Чанкинг документов
if not df.empty:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks_data = []
    for _, row in df.iterrows():
        text_chunks = splitter.split_text(row["text_full"])
        for chunk in text_chunks:
            chunks_data.append({"pmid": str(row["pmid"]), "text": chunk})

    chunks_df = pd.DataFrame(chunks_data)
    documents = chunks_df["text"].tolist()
    document_pmids = chunks_df["pmid"].tolist()

    print(f"Корпус загружен. Уникальных статей: {len(df)}")
    print(f"Количество чанков для индексации: {len(documents)}")
else:
    documents = []
    document_pmids = []

Загружен Golden Set (из JSON): 40 вопросов.
Корпус загружен. Уникальных статей: 10229
Количество чанков для индексации: 29718


## 2. Инициализация модели и получение эмбеддингов
Возьмем качественную модель из MTEB (например, `sergeyzh/rubert-mini-frida` или `sentence-transformers/all-MiniLM-L6-v2`). Будем использовать `all-MiniLM-L6-v2`, так как наш корпус (PubMed) на английском языке.

In [11]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Загрузка модели эмбеддингов: {model_name}...")
model = SentenceTransformer(model_name, device=device)
model.max_seq_length = 512 

print("Получение эмбеддингов чанков документов...")
embeddings = model.encode(documents, show_progress_bar=True, batch_size=32)

print("Получение эмбеддингов вопросов из Golden Set...")
queries = [item["question"] for item in golden_set]
query_vectors = model.encode(queries, show_progress_bar=True, batch_size=32)

d = embeddings.shape[1]
print(f"Размерность векторов: {d}")

flush_memory()

Загрузка модели эмбеддингов: sentence-transformers/all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2249.37it/s, Materializing param=pooler.dense.weight]                             


Получение эмбеддингов чанков документов...


Batches: 100%|██████████| 929/929 [00:38<00:00, 24.36it/s]


Получение эмбеддингов вопросов из Golden Set...


Batches: 100%|██████████| 2/2 [00:00<00:00, 169.89it/s]


Размерность векторов: 384


## 3. Векторная база данных (FAISS) и бенчмаркинг параметров

Для начала напишем функцию для замера времени поиска и вычисления "полноты" (Recall) относительно точного поиска (IndexFlatL2), чтобы оценивать качество приближенных индексов (IVF, HNSW).

In [12]:
def evaluate_index(index, query_vectors, k=5):
    if len(query_vectors) == 0:
        return 0.0, 0.0, []
        
    # Разогрев (прогреваем кэш процессора)
    index.search(query_vectors[:5], k)
    
    # Замер скорости
    start_time = time.perf_counter()
    similarities, ids = index.search(query_vectors, k)
    elapsed = time.perf_counter() - start_time
    
    hits = 0
    # Проверка на попадание правильного PMID в топ-k (Accuracy@k / Hit Rate)
    for i, item in enumerate(golden_set):
        gold_pmid = str(item["gold_pmid"])
        
        # Получаем pmid для найденных чанков (отсеиваем -1 на случай пустых результатов)
        retrieved_pmids = [str(document_pmids[idx]) for idx in ids[i] if idx != -1]
        
        if gold_pmid in retrieved_pmids:
            hits += 1
            
    accuracy = hits / len(golden_set) if len(golden_set) > 0 else 0.0
    return elapsed * 1000, accuracy, ids # Время в мс

# 1. Baseline: Точный поиск (FlatL2)
print("=== Точный поиск (FlatL2) ===")
flat_index = faiss.IndexFlatL2(d)
flat_index.add(embeddings)
time_flat, acc_flat, _ = evaluate_index(flat_index, query_vectors, k=5)
print(f"IndexFlatL2: Время поиска = {time_flat:.2f} мс, Accuracy@5 = {acc_flat:.3f}")

=== Точный поиск (FlatL2) ===
IndexFlatL2: Время поиска = 15.36 мс, Accuracy@5 = 0.950


### Индекс IVF (Inverted File)
Приближенный поиск, основанный на кластеризации пространства векторов на `nlist` кластеров. Во время поиска проверяются только `nprobe` ближайших кластеров. Чем больше `nprobe`, тем выше точность, но медленнее поиск.

In [13]:
print("\n=== Приближенный поиск (IVFFlat) ===")
# Количество кластеров. Эвристика: 4 * sqrt(N) или 8 * sqrt(N).
# Для 10000 документов (~30-40к чанков) sqrt(N) ~ 170-200. Возьмем 500, 1000, 2000 кластеров.
nlist_values = [500, 1000, 2000]
nprobe_values = [5, 10, 20, 50]

ivf_results = []

for nlist in nlist_values:
    # Инициализация и обучение индекса
    quantizer = faiss.IndexFlatL2(d)
    ivf_index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
    ivf_index.train(embeddings)
    ivf_index.add(embeddings)
    
    for nprobe in nprobe_values:
        ivf_index.nprobe = nprobe
        time_ivf, acc_ivf, _ = evaluate_index(ivf_index, query_vectors, k=5)
        ivf_results.append({
            "nlist": nlist,
            "nprobe": nprobe,
            "Time (ms)": time_ivf,
            "Accuracy@5": acc_ivf
        })

df_ivf = pd.DataFrame(ivf_results)
display(df_ivf)


=== Приближенный поиск (IVFFlat) ===


WARNING clustering 29718 points to 1000 centroids: please provide at least 39000 training points
WARNING clustering 29718 points to 2000 centroids: please provide at least 78000 training points


,nlist,nprobe,Time (ms),Accuracy@5
0,500,5,0.230886,0.825
1,500,10,0.255921,0.900
2,500,20,0.372037,0.950
3,500,50,0.805510,0.950
4,1000,5,0.320270,0.800
5,1000,10,0.261371,0.900
6,1000,20,0.337365,0.950
7,1000,50,0.566396,0.925
8,2000,5,0.348338,0.825
9,2000,10,0.363773,0.925


### Индекс HNSW (Hierarchical Navigable Small World)
Индекс на основе графов. `M` — количество связей на каждый узел графа (влияет на потребление памяти). `efConstruction` — параметр качества построения графа (влияет на время индексации). `efSearch` — параметр качества поиска (влияет на время поиска и Recall). Мы будем изменять `efSearch`.

In [14]:
print("\n=== Приближенный поиск (HNSWFlat) ===")
# M - количество связей. Рекомендуемые значения 16, 32, 64. Для большой базы возьмем побольше.
M_values = [16, 32, 64]
efSearch_values = [10, 20, 40, 80]

hnsw_results = []

for M in M_values:
    # Инициализация графа
    hnsw_index = faiss.IndexHNSWFlat(d, M)
    hnsw_index.hnsw.efConstruction = 64  # Увеличим для лучшего построения
    hnsw_index.add(embeddings)
    
    for efSearch in efSearch_values:
        hnsw_index.hnsw.efSearch = efSearch
        time_hnsw, acc_hnsw, _ = evaluate_index(hnsw_index, query_vectors, k=5)
        hnsw_results.append({
            "M": M,
            "efSearch": efSearch,
            "Time (ms)": time_hnsw,
            "Accuracy@5": acc_hnsw
        })

df_hnsw = pd.DataFrame(hnsw_results)
display(df_hnsw)


=== Приближенный поиск (HNSWFlat) ===


,M,efSearch,Time (ms),Accuracy@5
0,16,10,0.186990,0.800
1,16,20,0.197625,0.900
2,16,40,0.252582,0.925
3,16,80,0.424256,0.950
4,32,10,0.140556,0.925
5,32,20,0.152797,0.925
6,32,40,0.225546,0.950
7,32,80,0.392659,0.950
8,64,10,0.416532,0.950
9,64,20,0.421960,0.950


## 4. Ключевые выводы

1. **Точный поиск (IndexFlatL2):** 
   - Показывает эталонную метрику Accuracy@5, равную **0.950**. Это потолок качества для нашей модели эмбеддингов на данном Golden Set.
   - Скорость поиска на нашем датасете (~30 000 чанков) составила около **14.87 мс**. Хотя это время кажется небольшим, оно линейно зависит от размера базы (O(N)), и на миллионах документов такой подход станет серьезным "бутылочным горлышком".

2. **Приближенный поиск на основе кластеризации (IndexIVFFlat):**
   - Время поиска значительно сократилось (от 0.23 до 0.85 мс) по сравнению с точным поиском.
   - При малом количестве `nprobe` (например, 5) точность падает до 0.800–0.825.
   - При увеличении `nprobe` до 20 точность восстанавливается до эталонных **0.950**, а время поиска (например, при `nlist=1000`) составляет всего **0.34 мс**, что более чем в 40 раз быстрее точного поиска.

3. **Приближенный поиск на основе графов (IndexHNSWFlat):**
   - Графовый индекс HNSW также обеспечивает отличный баланс скорости и качества.
   - При малых параметрах (`M=16`, `efSearch=10`) поиск занимает всего **0.18 мс**, но точность снижается до 0.800.
   - При параметрах `M=32` и `efSearch=40` метрика достигает максимальных **0.950**, а время поиска составляет около **0.23 мс**. При избыточно больших параметрах (`M=64`) время поиска возрастает до 0.8-3 мс без дальнейшего прироста точности.

**Общий вывод:**
Использование индексов `IVF` и `HNSW` в FAISS позволяет ускорить поиск в десятки раз (с ~15 мс до долей миллисекунды) даже на таком небольшом корпусе в 30 тысяч чанков, сохраняя при этом точность поиска (0.950) на 100% относительно полного перебора (FlatL2). В условиях реального масштабирования (от сотен тысяч до сотен миллионов векторов) применение `HNSW` или настроенного `IVF` становится критически важным для обеспечения отзывчивости систем векторного поиска (например, в пайплайнах RAG).